# Refactoring Level Classification

**What this establishes.** Every refactoring type RefactoringMiner reports in this corpus is
assigned an abstraction level from Murphy-Hill, Parnin & Black, *How We Refactor, and How We Know
It*, IEEE TSE 38(1):5–18, 2012, §3.6.

**What it cannot establish.** The criterion is applied by a human, one type at a time. A notebook
cannot derive the mapping from first principles. What it can do is make the mapping inspectable and
test it three ways:

| check | question |
|---|---|
| **anchors** | does the mapping reproduce the levels the paper itself assigns? |
| **independent application** | does it agree with another team applying the same criterion? |
| **published distributions** | does the resulting instance distribution resemble prior corpora? |

Passing all three does not make the mapping correct. It makes it *reproducible, externally
comparable, and wrong in ways that would be visible.*

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require
WORK = setup()

## 1. The criterion

Verbatim from Murphy-Hill §3.6:

> **High level** refactorings are those that change the signatures of classes, methods, and fields;
> refactorings at this level include RENAME CLASS, MOVE STATIC FIELD, and ADD PARAMETER.
>
> **Medium level** refactorings are those that change the signatures of classes, methods, and
> fields and also significantly change blocks of code; this level includes EXTRACT METHOD, INLINE
> CONSTANT, and CONVERT ANONYMOUS TYPE TO NESTED TYPE.
>
> **Low level** refactorings are those that make changes to only blocks of code; low level
> refactorings include EXTRACT LOCAL VARIABLE, RENAME LOCAL VARIABLE, and ADD ASSERTION.

**How it was applied.** One question per type: *what does this operation declare, move or rewrite,
and is that element visible outside a method body?*

| element | visibility | level |
|---|---|---|
| class, interface, package, method, field, parameter, return type, annotation, modifier | external — callers may need to change | **High** |
| local variable, statement, expression, conditional, loop | internal — invisible outside the method | **Low** |
| creates or removes an external declaration **and** relocates or rewrites a block | both | **Medium** |

Every type carries the specific reason it was placed, so a reader can disagree with a particular
call without re-deriving the whole scheme.

**One case needs a definition of "signature".** Parameter names and modifiers appear in a method
declaration but the *Java Language Specification* §8.4.2 defines a signature as the method name plus
the number and types of its formal parameters — names and `final` are excluded. So Rename Parameter
and Add/Remove Parameter Modifier do not change the signature, and are Low. These carry
`source: derived_jls`.

**One name-matching hazard.** RefactoringMiner emits `Extract And Move Method`; refactoring
catalogues usually write `Extract and Move Method`. Exact-string matching silently drops twelve
types — 13,444 instances in this corpus, most of them Medium-level — so lookup is case-insensitive
and unmatched types are reported rather than ignored.

In [15]:
MAPPING = {
    "Add Attribute Annotation": [
        "High",
        "annotation on a field declaration",
        "derived"
    ],
    "Add Attribute Modifier": [
        "High",
        "modifier on a field declaration",
        "derived"
    ],
    "Add Class Annotation": [
        "High",
        "annotation on a class declaration",
        "derived"
    ],
    "Add Class Modifier": [
        "High",
        "modifier on a class declaration",
        "derived"
    ],
    "Add Method Annotation": [
        "High",
        "annotation on a method declaration",
        "derived"
    ],
    "Add Method Modifier": [
        "High",
        "modifier on a method declaration",
        "derived"
    ],
    "Add Parameter": [
        "High",
        "method signature; callers must change",
        "murphy_hill_figure_2"
    ],
    "Add Parameter Annotation": [
        "High",
        "annotation within a method signature",
        "derived"
    ],
    "Add Parameter Modifier": [
        "High",
        "Adds or removes `final` on a parameter: the method declaration changes and no block is touched at all. High for the same reason as the other parameter operations.",
        "derived"
    ],
    "Add Thrown Exception Type": [
        "High",
        "throws clause is part of the signature",
        "derived"
    ],
    "Add Variable Annotation": [
        "Low",
        "local variable declaration, inside a method body",
        "derived"
    ],
    "Add Variable Modifier": [
        "Low",
        "local variable declaration, inside a method body",
        "derived"
    ],
    "Assert Throws": [
        "Low",
        "rewrites assertion statements inside a test body",
        "derived"
    ],
    "Assert Timeout": [
        "Low",
        "rewrites assertion statements inside a test body",
        "derived"
    ],
    "Change Attribute Access Modifier": [
        "High",
        "field visibility, part of the declaration",
        "derived"
    ],
    "Change Attribute Type": [
        "Medium",
        "declaration changes and every use site in a block must follow",
        "derived"
    ],
    "Change Class Access Modifier": [
        "High",
        "class visibility, part of the declaration",
        "derived"
    ],
    "Change Method Access Modifier": [
        "High",
        "method visibility, part of the declaration",
        "murphy_hill_figure_2"
    ],
    "Change Parameter Type": [
        "Medium",
        "signature changes and the body using it must follow",
        "derived"
    ],
    "Change Return Type": [
        "Medium",
        "signature changes and return statements must follow",
        "derived"
    ],
    "Change Thrown Exception Type": [
        "High",
        "throws clause is part of the signature",
        "derived"
    ],
    "Change Type Declaration Kind": [
        "High",
        "class becomes interface or enum; the declaration itself",
        "derived"
    ],
    "Change Variable Type": [
        "Low",
        "local variable declaration, inside a method body",
        "derived"
    ],
    "Collapse Hierarchy": [
        "High",
        "merges two type declarations; member bodies unchanged",
        "derived"
    ],
    "Encapsulate Attribute": [
        "Medium",
        "adds accessor signatures AND rewrites every block that touched the field",
        "derived"
    ],
    "Extract Attribute": [
        "Medium",
        "creates a field declaration by lifting an expression out of a block",
        "derived"
    ],
    "Extract Class": [
        "High",
        "creates a class and relocates whole members; no block is rewritten",
        "murphy_hill_figure_2"
    ],
    "Extract Fixture": [
        "Medium",
        "creates a setup method and moves block content into it",
        "derived"
    ],
    "Extract Interface": [
        "High",
        "creates a type declaration from existing signatures",
        "derived"
    ],
    "Extract Method": [
        "Medium",
        "creates a method signature and moves a block into it",
        "murphy_hill_figure_2"
    ],
    "Extract Subclass": [
        "High",
        "creates a type declaration and relocates whole members",
        "derived"
    ],
    "Extract Superclass": [
        "High",
        "creates a type declaration and relocates whole members",
        "derived"
    ],
    "Extract Variable": [
        "Low",
        "introduces a local within a block",
        "murphy_hill_figure_2"
    ],
    "Extract and Move Method": [
        "Medium",
        "creates a method signature, moves a block, relocates the result",
        "derived"
    ],
    "Inline Attribute": [
        "Medium",
        "removes a field declaration and rewrites the blocks that used it",
        "derived"
    ],
    "Inline Method": [
        "Medium",
        "removes a method signature and folds its block into callers",
        "murphy_hill_figure_2"
    ],
    "Inline Variable": [
        "Low",
        "removes a local within a block",
        "murphy_hill_figure_2"
    ],
    "Invert Condition": [
        "Low",
        "rewrites a conditional expression",
        "derived"
    ],
    "Localize Parameter": [
        "Medium",
        "removes a parameter from the signature and adds a local in the block",
        "derived"
    ],
    "Merge Attribute": [
        "High",
        "combines field declarations",
        "derived"
    ],
    "Merge Catch": [
        "Low",
        "combines catch blocks",
        "derived"
    ],
    "Merge Class": [
        "High",
        "combines type declarations; member bodies unchanged",
        "derived"
    ],
    "Merge Conditional": [
        "Low",
        "combines conditional statements",
        "derived"
    ],
    "Merge Method": [
        "Medium",
        "removes method signatures and merges their blocks",
        "derived"
    ],
    "Merge Package": [
        "High",
        "package declarations only",
        "derived"
    ],
    "Merge Parameter": [
        "High",
        "method signature",
        "derived"
    ],
    "Merge Variable": [
        "Low",
        "combines locals within a block",
        "derived"
    ],
    "Modify Attribute Annotation": [
        "High",
        "annotation on a field declaration",
        "derived"
    ],
    "Modify Class Annotation": [
        "High",
        "annotation on a class declaration",
        "derived"
    ],
    "Modify Method Annotation": [
        "High",
        "annotation on a method declaration",
        "derived"
    ],
    "Modify Parameter Annotation": [
        "High",
        "annotation within a method signature",
        "derived"
    ],
    "Modify Variable Annotation": [
        "Low",
        "local variable declaration, inside a method body",
        "derived"
    ],
    "Move Annotation": [
        "High",
        "relocates an annotation between declarations",
        "derived"
    ],
    "Move Attribute": [
        "High",
        "relocates a field declaration",
        "murphy_hill_figure_2"
    ],
    "Move Class": [
        "High",
        "relocates a type declaration",
        "derived"
    ],
    "Move Code": [
        "Low",
        "relocates statements within or between bodies",
        "derived"
    ],
    "Move Method": [
        "High",
        "relocates a whole method; its body is unchanged",
        "murphy_hill_figure_2"
    ],
    "Move Package": [
        "High",
        "relocates a package declaration",
        "derived"
    ],
    "Move and Inline Method": [
        "Medium",
        "relocates a method then folds its block into the call site",
        "derived"
    ],
    "Move and Rename Attribute": [
        "High",
        "relocates and renames a field declaration",
        "derived"
    ],
    "Move and Rename Class": [
        "High",
        "relocates and renames a type declaration",
        "derived"
    ],
    "Move and Rename Method": [
        "High",
        "relocates and renames a whole method",
        "derived"
    ],
    "Parameterize Attribute": [
        "Medium",
        "replaces a field with a parameter; signature and block both change",
        "derived"
    ],
    "Parameterize Test": [
        "Medium",
        "adds a parameterised signature and rewrites the test body",
        "derived"
    ],
    "Parameterize Variable": [
        "Medium",
        "promotes a local to a parameter; signature and block both change",
        "murphy_hill_figure_2"
    ],
    "Pull Up Attribute": [
        "High",
        "relocates a field declaration up the hierarchy",
        "derived"
    ],
    "Pull Up Method": [
        "High",
        "relocates a whole method up the hierarchy",
        "derived"
    ],
    "Push Down Attribute": [
        "High",
        "relocates a field declaration down the hierarchy",
        "murphy_hill_figure_2"
    ],
    "Push Down Method": [
        "High",
        "relocates a whole method down the hierarchy",
        "murphy_hill_figure_2"
    ],
    "Remove Attribute Annotation": [
        "High",
        "annotation on a field declaration",
        "derived"
    ],
    "Remove Attribute Modifier": [
        "High",
        "modifier on a field declaration",
        "derived"
    ],
    "Remove Class Annotation": [
        "High",
        "annotation on a class declaration",
        "derived"
    ],
    "Remove Class Modifier": [
        "High",
        "modifier on a class declaration",
        "derived"
    ],
    "Remove Method Annotation": [
        "High",
        "annotation on a method declaration",
        "derived"
    ],
    "Remove Method Modifier": [
        "High",
        "modifier on a method declaration",
        "derived"
    ],
    "Remove Parameter": [
        "High",
        "method signature; callers must change",
        "murphy_hill_figure_2"
    ],
    "Remove Parameter Annotation": [
        "High",
        "annotation within a method signature",
        "derived"
    ],
    "Remove Parameter Modifier": [
        "High",
        "Adds or removes `final` on a parameter: the method declaration changes and no block is touched at all. High for the same reason as the other parameter operations.",
        "derived"
    ],
    "Remove Thrown Exception Type": [
        "High",
        "throws clause is part of the signature",
        "derived"
    ],
    "Remove Variable Annotation": [
        "Low",
        "local variable declaration, inside a method body",
        "derived"
    ],
    "Remove Variable Modifier": [
        "Low",
        "local variable declaration, inside a method body",
        "derived"
    ],
    "Rename Attribute": [
        "High",
        "field declaration; references must change",
        "murphy_hill_figure_2"
    ],
    "Rename Class": [
        "High",
        "type declaration; references must change",
        "murphy_hill_figure_2"
    ],
    "Rename Method": [
        "High",
        "method declaration; callers must change",
        "murphy_hill_figure_2"
    ],
    "Rename Package": [
        "High",
        "package declaration",
        "derived"
    ],
    "Rename Parameter": [
        "High",
        "Murphy-Hill's Low level requires changes to ONLY blocks of code, and this changes the method declaration line. Medium requires blocks to be SIGNIFICANTLY changed, and a rename is identifier substitution. By elimination, High -- consistent with Add, Remove and Reorder Parameter, all marked High in Figure 2.",
        "derived"
    ],
    "Rename Variable": [
        "Low",
        "local identifier, invisible outside the method",
        "murphy_hill_figure_2"
    ],
    "Reorder Parameter": [
        "High",
        "method signature; callers must change",
        "murphy_hill_figure_2"
    ],
    "Replace Anonymous with Class": [
        "Medium",
        "creates a type declaration and rewrites the block that held it",
        "derived"
    ],
    "Replace Anonymous with Lambda": [
        "Low",
        "rewrites an expression inside a block",
        "derived"
    ],
    "Replace Attribute": [
        "Medium",
        "substitutes one field declaration for another; every block referencing it must be rewritten to handle the new type or semantics, the same consequence that places Change Attribute Type at Medium",
        "derived"
    ],
    "Replace Attribute with Variable": [
        "Medium",
        "removes a field declaration, adds a local in the block",
        "derived"
    ],
    "Replace Conditional With Ternary": [
        "Low",
        "rewrites a conditional expression",
        "derived"
    ],
    "Replace Conditional with Assumption": [
        "Low",
        "rewrites a conditional statement in a test body",
        "derived"
    ],
    "Replace Generic With Diamond": [
        "Low",
        "rewrites a type argument in an expression",
        "derived"
    ],
    "Replace Loop with Pipeline": [
        "Low",
        "rewrites a loop as a stream expression",
        "derived"
    ],
    "Replace Pipeline with Loop": [
        "Low",
        "rewrites a stream expression as a loop",
        "derived"
    ],
    "Replace Variable with Attribute": [
        "Medium",
        "adds a field declaration, removes a local from the block",
        "derived"
    ],
    "Split Attribute": [
        "High",
        "one field declaration becomes several",
        "derived"
    ],
    "Split Class": [
        "High",
        "one type declaration becomes several; whole members relocate",
        "derived"
    ],
    "Split Conditional": [
        "Low",
        "divides a conditional statement",
        "derived"
    ],
    "Split Method": [
        "Medium",
        "one signature becomes several, and the block is divided among them",
        "derived"
    ],
    "Split Package": [
        "High",
        "one package declaration becomes several",
        "derived"
    ],
    "Split Parameter": [
        "High",
        "method signature",
        "derived"
    ],
    "Split Variable": [
        "Low",
        "one local becomes several within a block",
        "derived"
    ],
    "Try With Resources": [
        "Low",
        "restructures a try statement",
        "derived"
    ],
    "Move Source Folder": [
        "High",
        "relocates a source folder, moving whole package and type declarations; no block is rewritten",
        "derived"
    ]
}

import collections
import pandas as pd

# RefactoringMiner emits "Extract And Move Method" while catalogues commonly write
# "Extract and Move Method". Matching on the exact string silently drops those types --
# 13,444 instances in this corpus, most of them Medium-level. Lookup is case-insensitive.
NORM = {k.lower(): (k, v) for k, v in MAPPING.items()}

def level_of(rtype):
    """Level for a RefactoringMiner type name, or None if the type is unknown."""
    hit = NORM.get((rtype or "").strip().lower())
    return hit[1][0] if hit else None


print(f"{len(MAPPING)} refactoring types classified\n")
print(dict(collections.Counter(v[0] for v in MAPPING.values())))
print()
print("Types the PAPER classifies explicitly (used as anchors):\n")
for n, (lvl, why, src) in sorted(MAPPING.items()):
    if src == "murphy_hill_figure_2":
        print(f"   {n:<32}{lvl}")

107 refactoring types classified

{'High': 61, 'Low': 25, 'Medium': 21}

Types the PAPER classifies explicitly (used as anchors):

   Add Parameter                   High
   Change Method Access Modifier   High
   Extract Class                   High
   Extract Method                  Medium
   Extract Variable                Low
   Inline Method                   Medium
   Inline Variable                 Low
   Move Attribute                  High
   Move Method                     High
   Parameterize Variable           Medium
   Push Down Attribute             High
   Push Down Method                High
   Remove Parameter                High
   Rename Attribute                High
   Rename Class                    High
   Rename Method                   High
   Rename Variable                 Low
   Reorder Parameter               High


## 2. Check 1 — does it reproduce the paper's own assignments?

Murphy-Hill assign levels to specific refactorings in Figure 2 and §3.6. Those are the paper's own
judgements, not ours. If applying the criterion to those types produces a different answer, the
criterion is being applied wrongly.

Two of the paper's names differ from RefactoringMiner's vocabulary and are matched by meaning:
INTRODUCE PARAMETER is RefactoringMiner's *Parameterize Variable*, and EXTRACT LOCAL is *Extract
Variable*.

In [16]:
# The paper's own assignments, transcribed from Figure 2 and section 3.6.
PAPER = {
    "Extract Method": "Medium",            # section 3.6, explicit
    "Inline Method": "Medium",             # Figure 2
    "Extract Variable": "Low",             # Figure 2, "Extract Local (L)"
    "Inline Variable": "Low",              # Figure 2, "Inline Local (L)"
    "Rename Variable": "Low",              # Figure 2, "Rename Local (L)"
    "Rename Attribute": "High",            # Figure 2, "Rename Field (H)"
    "Rename Method": "High",               # Figure 2
    "Rename Class": "High",                # Figure 2, "Rename Type (H)"
    "Add Parameter": "High",               # Figure 2
    "Remove Parameter": "High",            # Figure 2
    "Reorder Parameter": "High",           # Figure 2
    "Extract Class": "High",               # Figure 2
    "Change Method Access Modifier": "High",   # Figure 2, "Increase/Decrease Method Visibility (H)"
    "Push Down Method": "High",            # Figure 2, "Push Down (H)"
    "Push Down Attribute": "High",         # Figure 2, "Push Down (H)"
    "Move Method": "High",                 # Figure 2, "Move Member (H)"
    "Move Attribute": "High",              # Figure 2, "Move Member (H)"
    "Parameterize Variable": "Medium",     # Figure 2, "Introduce Parameter (M)"
}

rows = []
for n, expected in sorted(PAPER.items()):
    got = MAPPING[n][0]
    rows.append({"type": n, "paper": expected, "ours": got, "match": got == expected})

A = pd.DataFrame(rows)
print(A.to_string(index=False))
n_ok = int(A["match"].sum())
print(f"\n{n_ok}/{len(A)} reproduce the paper's assignment")
if n_ok < len(A):
    print("\nMISMATCHES -- the criterion is being applied differently from the paper:")
    print(A[~A["match"]].to_string(index=False))

                         type  paper   ours  match
                Add Parameter   High   High   True
Change Method Access Modifier   High   High   True
                Extract Class   High   High   True
               Extract Method Medium Medium   True
             Extract Variable    Low    Low   True
                Inline Method Medium Medium   True
              Inline Variable    Low    Low   True
               Move Attribute   High   High   True
                  Move Method   High   High   True
        Parameterize Variable Medium Medium   True
          Push Down Attribute   High   High   True
             Push Down Method   High   High   True
             Remove Parameter   High   High   True
             Rename Attribute   High   High   True
                 Rename Class   High   High   True
                Rename Method   High   High   True
              Rename Variable    Low    Low   True
            Reorder Parameter   High   High   True

18/18 reproduce the paper's as

## 3. Check 2 — does an independent team agree?

Horikawa et al. (arXiv:2511.04824) applied the same Murphy-Hill criterion to RefactoringMiner's
types and published the result. Their mapping is fetched here and compared.

**This is a cross-check, not the source.** Murphy-Hill (TSE 2012) is peer-reviewed and is what the
classification cites; the arXiv preprint is used to test whether two independent applications of
the criterion converge.

They classified **103** types against the **106** in this corpus, so a direct reuse was never
available regardless.

In [17]:
import re, urllib.request, urllib.error

URL = ("https://raw.githubusercontent.com/Mont9165/Agent_Refactoring_Analysis/"
       "main/Refactoring_Level_Classification.md")

def fetch_horikawa(url=URL):
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            txt = r.read().decode("utf-8")
    except urllib.error.URLError as e:
        print(f"   could not fetch ({e.reason}); skipping this check")
        return {}, {}
    level, m, declared = None, {}, {}
    for line in txt.split("\n"):
        s = line.strip()
        h = re.match(r"^(High|Medium|Low)-level \([HML]\) Refactorings \((\d+) types\)", s)
        if h:
            level = h.group(1); declared[level] = int(h.group(2)); continue
        b = re.match(r"^\*\s+([^:]+):", s)
        if b and level:
            m[b.group(1).strip()] = level
    return m, declared


THEIRS, DECLARED = fetch_horikawa()

if THEIRS:
    parsed = collections.Counter(THEIRS.values())
    print(f"declared in their headers : {DECLARED}")
    print(f"actually listed           : {dict(parsed)}")
    if DECLARED and any(DECLARED.get(k) != parsed.get(k) for k in DECLARED):
        print("\n   note: their published file is internally inconsistent -- the section headers")
        print("   and the bullet lists disagree. The parsed lists are used here.")

    both = sorted(set(MAPPING) & set(THEIRS))
    agree = [n for n in both if MAPPING[n][0] == THEIRS[n]]
    disagree = [n for n in both if MAPPING[n][0] != THEIRS[n]]

    print(f"\ncomparable types : {len(both)}")
    print(f"agree            : {len(agree)}  ({len(agree)/len(both)*100:.1f}%)")
    print(f"disagree         : {len(disagree)}")
    print(f"only in ours     : {len(set(MAPPING) - set(THEIRS))} "
          f"-- {', '.join(sorted(set(MAPPING) - set(THEIRS)))}")

    D = pd.DataFrame([{"type": n, "ours": MAPPING[n][0], "horikawa": THEIRS[n],
                       "reason we placed it here": MAPPING[n][1]} for n in disagree])
    print("\nDISAGREEMENTS\n")
    print(D.to_string(index=False))

declared in their headers : {'High': 58, 'Medium': 21, 'Low': 24}
actually listed           : {'High': 56, 'Medium': 19, 'Low': 28}

   note: their published file is internally inconsistent -- the section headers
   and the bullet lists disagree. The parsed lists are used here.

comparable types : 102
agree            : 93  (91.2%)
disagree         : 9
only in ours     : 5 -- Extract Fixture, Move Annotation, Move Source Folder, Replace Attribute, Replace Conditional with Assumption

DISAGREEMENTS

                     type   ours horikawa                                                                                                                                                                                                                                                                                            reason we placed it here
   Add Parameter Modifier   High      Low                                                                                                         

### Reading the disagreements

Three groups, and they carry different weight.

**Backed by the paper (4).** EXTRACT CLASS is marked High in Figure 2; Extract Subclass and Split
Class follow the same reasoning — whole members relocate and no method body is rewritten.
Parameterize Variable is Medium because the paper marks INTRODUCE PARAMETER as Medium. Here the
paper decides, and check 1 above verifies it.

**Follows from the definition (2).** Encapsulate Attribute and Parameterize Test each add a
signature *and* rewrite blocks, which is what Medium means.

**Genuinely ambiguous (3).** Rename Parameter, Add Parameter Modifier, Remove Parameter Modifier.
These turn on whether *signature* means the textual declaration or the caller-visible contract. A
parameter name and a `final` modifier appear in the declaration but cannot break a caller. Placed
High here, following the other parameter operations; placed Low by Horikawa et al. on the
caller-visibility reading.

Neither reading is wrong. Section 5 measures whether the choice changes anything.

## 4. Check 3 — the distribution over actual refactoring instances

The mapping is applied to every refactoring RefactoringMiner detected in this corpus, and compared
against two published distributions.

A close match is weak evidence — corpora differ — but a wildly different one would indicate either
a mapping error or a corpus unlike anything previously studied. Either is worth knowing.

In [18]:
import json, zipfile, urllib.request
from pathlib import Path

RECORD = "24182007"   # the thesis corpus, public: doi.org/10.5281/zenodo.24182007
REPOS  = ["commons-lang", "checkstyle", "dubbo", "junit-framework", "questdb"]
DATA   = Path("corpus"); DATA.mkdir(exist_ok=True)   # shared with 05a_assemble


def fetch_corpus():
    need = [r for r in REPOS if not any(DATA.rglob(f"refactorings_{r}.jsonl"))]
    if not need:
        print("corpus already extracted"); return
    with urllib.request.urlopen(f"https://zenodo.org/api/records/{RECORD}", timeout=60) as r:
        meta = json.load(r)
    wanted = {f"{r}_mined.zip" for r in need}    # the record also holds git bundles; skip them
    for f in meta.get("files", []):
        name = f.get("key") or f.get("filename")
        if name not in wanted:
            continue
        dest = DATA / name
        if not dest.exists():
            url = f["links"].get("self") or f["links"].get("download")
            print(f"   {name} ({f.get('size', 0)/1e6:.0f} MB) ...", end=" ", flush=True)
            urllib.request.urlretrieve(url, dest); print("done")
        with zipfile.ZipFile(dest) as z:
            z.extractall(DATA / name[:-4])


fetch_corpus()

def instance_counts(repo):
    """Every detected refactoring instance in a repo, counted by type."""
    hits = sorted(DATA.rglob(f"refactorings_{repo}.jsonl"))
    if not hits:
        return collections.Counter()
    c = collections.Counter()
    with open(hits[0], encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            for r in rec.get("refactorings") or []:
                t = r.get("type") or r.get("name")
                if t:
                    c[t] += 1
    return c


COUNTS = {r: instance_counts(r) for r in REPOS}
ALL = collections.Counter()
for c in COUNTS.values():
    ALL.update(c)

unmapped = {t: n for t, n in ALL.items() if level_of(t) is None}
print(f"\n{sum(ALL.values()):,} refactoring instances, {len(ALL)} distinct types")
if unmapped:
    print(f"\nUNMAPPED TYPES ({len(unmapped)}) -- these must be classified before use:")
    for t, n in sorted(unmapped.items(), key=lambda x: -x[1]):
        print(f"   {t:<44}{n:>8}")
else:
    print("every detected type is classified")

corpus already extracted

278,923 refactoring instances, 106 distinct types
every detected type is classified


In [19]:
rows = []
for repo, c in COUNTS.items():
    lv = collections.Counter()
    for t, n in c.items():
        l = level_of(t)
        if l:
            lv[l] += n
    tot = sum(lv.values())
    if tot:
        rows.append({"repo": repo, "instances": tot,
                     **{k: round(lv[k] / tot * 100, 1) for k in ("High", "Medium", "Low")}})

corpus_lv = collections.Counter()
for t, n in ALL.items():
    l = level_of(t)
    if l:
        corpus_lv[l] += n
tot = sum(corpus_lv.values())
rows.append({"repo": "— this corpus —", "instances": tot,
             **{k: round(corpus_lv[k] / tot * 100, 1) for k in ("High", "Medium", "Low")}})
rows.append({"repo": "Murphy-Hill 2012 (Eclipse CVS)", "instances": None,
             "High": 60.0, "Medium": 22.0, "Low": 18.0})
rows.append({"repo": "Horikawa 2025 (human)", "instances": None,
             "High": 54.9, "Medium": 20.7, "Low": 24.4})

print("SHARE OF REFACTORING INSTANCES BY LEVEL (%)\n")
print(pd.DataFrame(rows).set_index("repo").to_string())

SHARE OF REFACTORING INSTANCES BY LEVEL (%)

                                instances  High  Medium   Low
repo                                                         
commons-lang                      37809.0  62.1    11.0  26.9
checkstyle                        41102.0  61.6    11.1  27.3
dubbo                             55525.0  63.4    19.0  17.6
junit-framework                   41570.0  60.9    17.3  21.8
questdb                          102917.0  56.3    25.3  18.4
— this corpus —                  278923.0  60.0    18.8  21.2
Murphy-Hill 2012 (Eclipse CVS)        NaN  60.0    22.0  18.0
Horikawa 2025 (human)                 NaN  54.9    20.7  24.4


## 5. Sensitivity — every disagreement at once

Nine types are classified differently here than by Horikawa et al. (§3). Each was placed for a
stated reason, but a reason is not a proof, and the honest question is whether the choices matter.

The distribution is recomputed with **all nine flipped to their assignments** — the strongest
alternative reading available, since it is another team's complete application of the same
criterion. If the corpus looks much the same either way, the disagreements are bookkeeping. If it
moves, every level-dependent result downstream has to be reported both ways.

The nine fall into three groups, and they are not equally uncertain:

| group | types | how contestable |
|---|---|---|
| **settled by the paper** | Extract Class, Extract Subclass, Split Class, Parameterize Variable | Figure 2 assigns two of these directly; the 18/18 anchor check in §2 confirms the reading |
| **follows from the definition** | Encapsulate Attribute, Parameterize Test | both add a signature *and* rewrite blocks, which is what Medium means |
| **parameter family** | Rename Parameter, Add/Remove Parameter Modifier | changes the declaration line, so Low ("only blocks") cannot apply |

The list is read from §3 rather than hard-coded, so it stays correct if the mapping changes.

In [20]:
# The contested set comes from the comparison in section 3, so it cannot drift out of date.
try:
    CONTESTED = {n: THEIRS[n] for n in disagree}
except NameError:
    CONTESTED = {}          # section 3 could not fetch; nothing to compare against

if not CONTESTED:
    print("No comparison available -- section 3 did not fetch the Horikawa mapping.")
else:
    alt = dict(MAPPING)
    for t, their_level in CONTESTED.items():
        alt[t] = [their_level, MAPPING[t][1], MAPPING[t][2]]
    ALT_NORM = {k.lower(): (k, v) for k, v in alt.items()}

    def dist(norm):
        lv = collections.Counter()
        for t, n in ALL.items():
            hit = norm.get(t.strip().lower())
            if hit:
                lv[hit[1][0]] += n
        tot = sum(lv.values())
        return {k: round(lv[k] / tot * 100, 1) for k in ("High", "Medium", "Low")}, tot

    def count_of(name):
        return sum(v for k, v in ALL.items() if k.strip().lower() == name.lower())

    base, tot = dist(NORM)
    moved, _ = dist(ALT_NORM)

    # ---- what each disagreement is worth, in instances ----
    GROUPS = {
        "settled by the paper": ["Extract Class", "Extract Subclass", "Split Class",
                                 "Parameterize Variable"],
        "follows from the definition": ["Encapsulate Attribute", "Parameterize Test"],
        "parameter family": ["Rename Parameter", "Add Parameter Modifier",
                             "Remove Parameter Modifier"],
    }
    group_of = {t: g for g, ts in GROUPS.items() for t in ts}

    rows = []
    for t, their in sorted(CONTESTED.items(), key=lambda x: -count_of(x[0])):
        n = count_of(t)
        rows.append({"type": t, "ours": MAPPING[t][0], "theirs": their,
                     "instances": n, "% of corpus": round(n / tot * 100, 2),
                     "group": group_of.get(t, "—")})
    C = pd.DataFrame(rows)
    print("EACH DISAGREEMENT, BY WEIGHT IN THIS CORPUS\n")
    print(C.to_string(index=False))

    # ---- corpus-level effect ----
    print("\n\nDISTRIBUTION UNDER BOTH READINGS\n")
    print(f"{'':<12}{'as classified':>15}{'all 9 -> theirs':>18}{'shift':>9}")
    print("-" * 55)
    for k in ("High", "Medium", "Low"):
        print(f"{k:<12}{base[k]:>14.1f}%{moved[k]:>17.1f}%{moved[k]-base[k]:>+8.1f}")

    contested_n = sum(count_of(t) for t in CONTESTED)
    print(f"\ninstances in contested types: {contested_n:,} of {tot:,} "
          f"({contested_n/tot*100:.2f}%)")

    # ---- how much of the movement each group is responsible for ----
    print("\n\nCONTRIBUTION BY GROUP\n")
    for g, ts in GROUPS.items():
        sub = dict(MAPPING)
        for t in ts:
            if t in CONTESTED:
                sub[t] = [CONTESTED[t], MAPPING[t][1], MAPPING[t][2]]
        d, _ = dist({k.lower(): (k, v) for k, v in sub.items()})
        worst = max(abs(d[k] - base[k]) for k in ("High", "Medium", "Low"))
        n = sum(count_of(t) for t in ts if t in CONTESTED)
        print(f"   {g:<30}{n:>8,} instances   max shift {worst:>5.1f} pp")

    # ---- per repository, since downstream analysis is per repository ----
    print("\n\nPER REPOSITORY (High %, both readings)\n")
    prows = []
    for repo, c in COUNTS.items():
        def rd(norm):
            lv = collections.Counter()
            for t, n in c.items():
                hit = norm.get(t.strip().lower())
                if hit:
                    lv[hit[1][0]] += n
            s = sum(lv.values())
            return round(lv["High"] / s * 100, 1) if s else float("nan")
        a, b = rd(NORM), rd(ALT_NORM)
        prows.append({"repo": repo, "ours": a, "theirs": b, "shift": round(b - a, 1)})
    print(pd.DataFrame(prows).set_index("repo").to_string())

    shift = max(abs(moved[k] - base[k]) for k in ("High", "Medium", "Low"))
    print()
    if shift < 2:
        print(f"Largest shift is {shift:.1f} percentage points across all nine disagreements")
        print("simultaneously. The classification choices do not change the picture, and recording")
        print("them is sufficient.")
    elif shift < 5:
        print(f"Largest shift is {shift:.1f} percentage points. Modest, but level-dependent results")
        print("should note the sensitivity.")
    else:
        print(f"Largest shift is {shift:.1f} percentage points. The classification is load-bearing:")
        print("report level-dependent results under both readings.")

EACH DISAGREEMENT, BY WEIGHT IN THIS CORPUS

                     type   ours theirs  instances  % of corpus                       group
         Rename Parameter   High    Low      10238         3.67            parameter family
   Add Parameter Modifier   High    Low       5452         1.95            parameter family
    Parameterize Variable Medium   High       1605         0.58        settled by the paper
            Extract Class   High Medium       1117         0.40        settled by the paper
    Encapsulate Attribute Medium   High        876         0.31 follows from the definition
Remove Parameter Modifier   High    Low        481         0.17            parameter family
         Extract Subclass   High Medium        144         0.05        settled by the paper
              Split Class   High Medium        124         0.04        settled by the paper
        Parameterize Test Medium   High         50         0.02 follows from the definition


DISTRIBUTION UNDER BOTH READINGS


## 6. What this justifies

The mapping is not asserted. It is:

- **derived from a peer-reviewed criterion**, quoted in §1
- **verified against that paper's own assignments** for the 18 types it classifies (§2)
- **cross-checked against an independent application** of the same criterion (§3)
- **compared against two published instance distributions** (§4)
- **tested for sensitivity** on the types where the criterion is genuinely ambiguous (§5)

The disagreements are reported rather than reconciled. Two independent applications of the same
published criterion agreeing on most types is evidence the criterion is applicable; where they
differ, the reason is recorded and the reader can choose.

The full mapping with per-type justification is written below, and is the artifact everything
downstream reads.

In [21]:
import yaml
from datetime import date

doc = {
    "meta": {
        "generated": str(date.today()),
        "types": len(MAPPING),
        "scheme": ("Murphy-Hill, Parnin & Black, 'How We Refactor, and How We Know It', "
                   "IEEE TSE 38(1):5-18, 2012, section 3.6"),
        "distribution_of_types": dict(collections.Counter(v[0] for v in MAPPING.values())),
        "anchors_reproduced": f"{n_ok}/{len(A)}",
        "contested_with_horikawa": CONTESTED,
        "jls_grounded": [n for n, v in MAPPING.items() if v[2] == "derived_jls"],
    },
    "refactorings": [
        {"name": n, "level": lvl, "criterion": why, "source": src}
        for n, (lvl, why, src) in sorted(MAPPING.items())
    ],
}
out = "refactoring_levels.yaml"
with open(out, "w", encoding="utf-8") as f:
    yaml.dump(doc, f, sort_keys=False, allow_unicode=True, width=100)
print(f"   {out} written -- {len(MAPPING)} types")

   refactoring_levels.yaml written -- 107 types
